# Advanced Problems with Solutions: Python 3.10 Pattern Matching and Strict Zip

This notebook contains advanced exercises based on two Python 3.10 topics:

- Structural pattern matching with `match` / `case`
- `zip(..., strict=True)` for validating equal-length iterables

Each problem includes:
- a realistic task,
- clear constraints,
- a reference solution,
- runnable tests,
- best-practice notes.

> Requires Python 3.10 or newer.

In [2]:
import sys
from dataclasses import dataclass
from typing import Any, Iterable, Iterator, Sequence

assert sys.version_info >= (3, 10), "This notebook requires Python 3.10+"

## Problem 1 — Build a Safe Robot Command Interpreter

You are given commands for a small robot. Commands may be represented in several forms:

```python
["move", "F", "L", "R"]
("move", "B")
{"cmd": "move", "directions": ["F", "F", "L"]}
"pick"
"drop"
```

Write `compile_command(command)` that returns a tuple of symbolic actions.

Rules:

- `"F"`, `"B"`, `"L"`, `"R"` map to arrows.
- `"pick"` and `"drop"` map to item actions.
- invalid commands must raise `ValueError`.
- invalid directions must raise `ValueError`, not `KeyError`.
- empty move commands such as `["move"]` are invalid.
- use structural pattern matching.

In [3]:
SYMBOLS = {
    "F": "→",
    "B": "←",
    "L": "↑",
    "R": "↓",
    "pick": "⤣",
    "drop": "⤥",
}

VALID_DIRECTIONS = {"F", "B", "L", "R"}


def compile_command(command: Any) -> tuple[str, ...]:
    """Compile one robot command into a tuple of symbolic actions."""
    match command:
        case "pick":
            return (SYMBOLS["pick"],)

        case "drop":
            return (SYMBOLS["drop"],)

        case ["move", *directions] if directions and set(directions) <= VALID_DIRECTIONS:
            return tuple(SYMBOLS[direction] for direction in directions)

        case ("move", *directions) if directions and set(directions) <= VALID_DIRECTIONS:
            return tuple(SYMBOLS[direction] for direction in directions)

        case {"cmd": "move", "directions": [*directions]} if directions and set(directions) <= VALID_DIRECTIONS:
            return tuple(SYMBOLS[direction] for direction in directions)

        case _:
            raise ValueError(f"Invalid command: {command!r}")

In [4]:
assert compile_command(["move", "F", "L", "R"]) == ("→", "↑", "↓")
assert compile_command(("move", "B")) == ("←",)
assert compile_command({"cmd": "move", "directions": ["F", "F", "L"]}) == ("→", "→", "↑")
assert compile_command("pick") == ("⤣",)
assert compile_command("drop") == ("⤥",)

for bad in (
    ["move"],
    ["move", "up"],
    {"cmd": "move", "directions": []},
    {"cmd": "move", "directions": ["F", "up"]},
    "fly",
):
    try:
        compile_command(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected ValueError for {bad!r}")

print("Problem 1 tests passed.")

Problem 1 tests passed.


### Solution Notes

This solution uses several important best practices:

- It validates captured values in guards instead of letting dictionary lookups raise accidental `KeyError`.
- It keeps the wildcard case last.
- It uses `set(directions) <= VALID_DIRECTIONS` to ensure every direction is legal.
- It returns a consistent type: always `tuple[str, ...]`.

## Problem 2 — Parse a Tiny Expression Language

Create an evaluator for nested expression tuples.

Supported forms:

```python
("lit", 10)
("neg", expression)
("add", left, right)
("sub", left, right)
("mul", left, right)
("div", left, right)
("let", name, value_expression, body_expression)
("var", name)
```

Example:

```python
("let", "x", ("lit", 10), ("mul", ("var", "x"), ("add", ("lit", 2), ("lit", 3))))
```

should evaluate to `50`.

Rules:

- use structural pattern matching;
- support nested expressions;
- raise `NameError` for unknown variables;
- raise `ValueError` for malformed expressions;
- raise `ZeroDivisionError` naturally for division by zero.

In [5]:
Env = dict[str, float]


def evaluate(expr: Any, env: Env | None = None) -> float:
    """Evaluate a tiny expression language represented as nested tuples."""
    if env is None:
        env = {}

    match expr:
        case ("lit", int(value) | float(value)):
            return value

        case ("var", str(name)):
            try:
                return env[name]
            except KeyError as exc:
                raise NameError(f"Unknown variable: {name}") from exc

        case ("neg", subexpr):
            return -evaluate(subexpr, env)

        case ("add", left, right):
            return evaluate(left, env) + evaluate(right, env)

        case ("sub", left, right):
            return evaluate(left, env) - evaluate(right, env)

        case ("mul", left, right):
            return evaluate(left, env) * evaluate(right, env)

        case ("div", left, right):
            return evaluate(left, env) / evaluate(right, env)

        case ("let", str(name), value_expr, body_expr):
            # Copy the environment so bindings do not leak outward.
            inner_env = env | {name: evaluate(value_expr, env)}
            return evaluate(body_expr, inner_env)

        case _:
            raise ValueError(f"Malformed expression: {expr!r}")

In [6]:
expr = (
    "let",
    "x",
    ("lit", 10),
    ("mul", ("var", "x"), ("add", ("lit", 2), ("lit", 3))),
)

assert evaluate(expr) == 50
assert evaluate(("neg", ("lit", 8))) == -8
assert evaluate(("sub", ("lit", 10), ("lit", 4))) == 6
assert evaluate(("div", ("lit", 9), ("lit", 3))) == 3

try:
    evaluate(("var", "missing"))
except NameError:
    pass
else:
    raise AssertionError("Expected NameError")

try:
    evaluate(("pow", ("lit", 2), ("lit", 10)))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

print("Problem 2 tests passed.")

Problem 2 tests passed.


### Solution Notes

This problem demonstrates nested decomposition, type patterns, OR patterns, and controlled error handling.

The `case ("lit", int(value) | float(value)):` branch is intentionally strict: it accepts numeric literals and rejects malformed literals such as `("lit", "10")`.

## Problem 3 — Validate Batched Training Data with `zip(strict=True)`

You are preparing supervised-learning training data.

You receive three iterables:

```python
features
labels
weights
```

Write `make_training_rows(features, labels, weights)` that yields dictionaries:

```python
{"x": feature, "y": label, "weight": weight}
```

Rules:

- use `zip(..., strict=True)`;
- do not convert input iterables to lists;
- preserve laziness;
- if lengths differ, allow Python to raise `ValueError`;
- reject non-positive weights with `ValueError`.

In [7]:
def make_training_rows(
    features: Iterable[Any],
    labels: Iterable[Any],
    weights: Iterable[float],
) -> Iterator[dict[str, Any]]:
    """Yield validated training rows lazily."""
    for feature, label, weight in zip(features, labels, weights, strict=True):
        if weight <= 0:
            raise ValueError(f"Weight must be positive, got {weight!r}")
        yield {"x": feature, "y": label, "weight": weight}

In [8]:
rows = list(make_training_rows(["a", "b"], [0, 1], [1.0, 0.5]))
assert rows == [
    {"x": "a", "y": 0, "weight": 1.0},
    {"x": "b", "y": 1, "weight": 0.5},
]

try:
    list(make_training_rows(["a", "b", "c"], [0, 1], [1.0, 1.0]))
except ValueError as exc:
    assert "shorter" in str(exc) or "longer" in str(exc)
else:
    raise AssertionError("Expected ValueError from zip(strict=True)")

try:
    list(make_training_rows(["a"], [1], [0]))
except ValueError as exc:
    assert "positive" in str(exc)
else:
    raise AssertionError("Expected ValueError for non-positive weight")

print("Problem 3 tests passed.")

Problem 3 tests passed.


### Solution Notes

`zip(strict=True)` is preferred here because silently truncating mismatched training data would create subtle bugs. The function is a generator, so it works with streams and large datasets.

## Problem 4 — Route API Events with Mapping Patterns

You receive API events as dictionaries.

Supported events:

```python
{"type": "user.created", "payload": {"id": int, "email": str}}
{"type": "user.deleted", "payload": {"id": int}}
{"type": "order.placed", "payload": {"id": int, "user_id": int, "items": list}}
{"type": "healthcheck"}
```

Write `route_event(event)` that returns a normalized tuple:

```python
("create_user", user_id, email)
("delete_user", user_id)
("place_order", order_id, user_id, item_count)
("ok",)
```

Rules:

- use mapping patterns;
- ignore extra fields;
- reject empty `items`;
- raise `ValueError` for unknown or malformed events.

In [9]:
def route_event(event: Any) -> tuple[Any, ...]:
    """Normalize API event dictionaries using mapping patterns."""
    match event:
        case {"type": "user.created", "payload": {"id": int(user_id), "email": str(email)}}:
            return ("create_user", user_id, email)

        case {"type": "user.deleted", "payload": {"id": int(user_id)}}:
            return ("delete_user", user_id)

        case {
            "type": "order.placed",
            "payload": {"id": int(order_id), "user_id": int(user_id), "items": [_, *rest]},
        }:
            return ("place_order", order_id, user_id, 1 + len(rest))

        case {"type": "healthcheck"}:
            return ("ok",)

        case _:
            raise ValueError(f"Unknown or malformed event: {event!r}")

In [10]:
assert route_event({"type": "healthcheck", "ts": 123}) == ("ok",)
assert route_event({"type": "user.created", "payload": {"id": 7, "email": "a@example.com", "role": "admin"}}) == (
    "create_user",
    7,
    "a@example.com",
)
assert route_event({"type": "user.deleted", "payload": {"id": 7}}) == ("delete_user", 7)
assert route_event({"type": "order.placed", "payload": {"id": 10, "user_id": 7, "items": ["book", "pen"]}}) == (
    "place_order",
    10,
    7,
    2,
)

for bad in (
    {"type": "order.placed", "payload": {"id": 10, "user_id": 7, "items": []}},
    {"type": "user.created", "payload": {"id": "7", "email": "a@example.com"}},
    {"type": "unknown"},
):
    try:
        route_event(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected ValueError for {bad!r}")

print("Problem 4 tests passed.")

Problem 4 tests passed.


### Solution Notes

Mapping patterns are excellent for JSON-like input. They match required keys while naturally allowing extra keys. The order event uses `[_, *rest]` to require at least one item.

## Problem 5 — Implement a Message Dispatcher with Class Patterns

Create a small message dispatcher for dataclass messages.

Messages:

```python
TextMessage(sender, text)
ImageMessage(sender, url, width, height)
Reaction(sender, emoji)
```

Write `summarize_message(message)`.

Rules:

- short text messages return `"sender: text"`;
- text longer than 20 characters is truncated to 17 characters plus `"..."`;
- square images return `"sender sent a square image: url"`;
- landscape images return `"sender sent a landscape image: url"`;
- portrait images return `"sender sent a portrait image: url"`;
- reactions return `"sender reacted with emoji"`;
- malformed or unknown messages raise `TypeError`;
- use class patterns and guards.

In [11]:
@dataclass(frozen=True)
class TextMessage:
    sender: str
    text: str


@dataclass(frozen=True)
class ImageMessage:
    sender: str
    url: str
    width: int
    height: int


@dataclass(frozen=True)
class Reaction:
    sender: str
    emoji: str


def summarize_message(message: Any) -> str:
    """Summarize a chat message using class patterns and guards."""
    match message:
        case TextMessage(str(sender), str(text)) if len(text) <= 20:
            return f"{sender}: {text}"

        case TextMessage(str(sender), str(text)):
            return f"{sender}: {text[:17]}..."

        case ImageMessage(str(sender), str(url), int(width), int(height)) if width <= 0 or height <= 0:
            raise TypeError(f"Invalid image dimensions: {width}x{height}")

        case ImageMessage(str(sender), str(url), int(width), int(height)) if width == height:
            return f"{sender} sent a square image: {url}"

        case ImageMessage(str(sender), str(url), int(width), int(height)) if width > height:
            return f"{sender} sent a landscape image: {url}"

        case ImageMessage(str(sender), str(url), int(width), int(height)):
            return f"{sender} sent a portrait image: {url}"

        case Reaction(str(sender), str(emoji)):
            return f"{sender} reacted with {emoji}"

        case _:
            raise TypeError(f"Unsupported message: {message!r}")

In [12]:
assert summarize_message(TextMessage("Ada", "Hello")) == "Ada: Hello"
assert summarize_message(TextMessage("Ada", "This message is definitely long")) == "Ada: This message is d..."
assert summarize_message(ImageMessage("Linus", "img://1", 100, 100)) == "Linus sent a square image: img://1"
assert summarize_message(ImageMessage("Linus", "img://2", 200, 100)) == "Linus sent a landscape image: img://2"
assert summarize_message(ImageMessage("Linus", "img://3", 100, 200)) == "Linus sent a portrait image: img://3"
assert summarize_message(Reaction("Grace", "👍")) == "Grace reacted with 👍"

try:
    summarize_message(ImageMessage("Linus", "bad", 0, 100))
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for invalid dimensions")

try:
    summarize_message({"sender": "Ada", "text": "Hello"})
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for unsupported message")

print("Problem 5 tests passed.")

Problem 5 tests passed.


### Solution Notes

Class patterns are useful when input is structured with classes or dataclasses. The invalid-dimensions case appears before the general image cases so that bad data is not accidentally summarized as valid.

## Problem 6 — Transpose a Rectangular Matrix Safely

Write `transpose_strict(matrix)`.

Rules:

- accept any iterable of row iterables;
- return a list of tuples representing columns;
- use `zip(..., strict=True)` to reject ragged matrices;
- do not silently truncate;
- return an empty list for an empty matrix.

In [13]:
from itertools import chain


def transpose_strict(matrix: Iterable[Iterable[Any]]) -> list[tuple[Any, ...]]:
    """Transpose a rectangular matrix and reject ragged rows."""
    rows = iter(matrix)

    try:
        first_row = tuple(next(rows))
    except StopIteration:
        return []

    return list(zip(first_row, *rows, strict=True))

In [14]:
assert transpose_strict([[1, 2, 3], [4, 5, 6]]) == [(1, 4), (2, 5), (3, 6)]
assert transpose_strict([]) == []
assert transpose_strict([[], []]) == []

try:
    transpose_strict([[1, 2], [3]])
except ValueError as exc:
    assert "shorter" in str(exc) or "longer" in str(exc)
else:
    raise AssertionError("Expected ValueError for ragged matrix")

try:
    transpose_strict([[], [1]])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for ragged matrix with empty first row")

print("Problem 6 tests passed.")

Problem 6 tests passed.


### Solution Notes

Regular `zip(*matrix)` silently truncates ragged rows. With `strict=True`, malformed matrix data fails fast instead of producing incomplete columns.

## Problem 7 — Compile a Mini Robot Program

You now receive a full robot program:

```python
[
    ["move", "F", "F"],
    "pick",
    {"cmd": "move", "directions": ["L", "R"]},
    "drop"
]
```

Write `compile_program(program)` that returns one flat tuple of symbols.

Rules:

- reuse `compile_command`;
- validate the program itself using pattern matching;
- an empty program is valid and compiles to `()`;
- a non-sequence program raises `ValueError`;
- if any command is invalid, raise a `ValueError` that includes the command index.

In [15]:
def compile_program(program: Any) -> tuple[str, ...]:
    """Compile a sequence of robot commands into one flat tuple of symbolic actions."""
    match program:
        case [*commands]:
            output: list[str] = []
            for index, command in enumerate(commands):
                try:
                    output.extend(compile_command(command))
                except ValueError as exc:
                    raise ValueError(f"Invalid command at index {index}: {command!r}") from exc
            return tuple(output)

        case _:
            raise ValueError(f"Program must be a sequence of commands, got {program!r}")

In [16]:
program = [
    ["move", "F", "F"],
    "pick",
    {"cmd": "move", "directions": ["L", "R"]},
    "drop",
]

assert compile_program(program) == ("→", "→", "⤣", "↑", "↓", "⤥")
assert compile_program([]) == ()

try:
    compile_program(123)
except ValueError as exc:
    assert "Program must be" in str(exc)
else:
    raise AssertionError("Expected ValueError for non-sequence program")

try:
    compile_program([["move", "F"], ["move", "up"]])
except ValueError as exc:
    assert "index 1" in str(exc)
else:
    raise AssertionError("Expected indexed ValueError")

print("Problem 7 tests passed.")

Problem 7 tests passed.


### Solution Notes

The function separates two concerns:

1. `compile_command` validates and compiles one command.
2. `compile_program` validates the outer structure and adds context to command errors.

Adding the command index is a debugging best practice because it tells the caller where the bad command occurred.

## Problem 8 — Align Event Streams Without Losing Data

Two services emit event IDs and timestamps separately.

Write `align_events(event_ids, timestamps)` that yields:

```python
{"event_id": event_id, "timestamp": timestamp}
```

Rules:

- use `zip(..., strict=True)`;
- preserve laziness;
- validate each event ID with pattern matching:
  - valid event IDs are strings shaped like `"evt_" + non-empty suffix`;
- validate each timestamp:
  - valid timestamps are positive integers;
- raise `ValueError` for invalid values.

In [17]:
def align_events(event_ids: Iterable[Any], timestamps: Iterable[Any]) -> Iterator[dict[str, Any]]:
    """Align event IDs and timestamps without silent truncation."""
    for event_id, timestamp in zip(event_ids, timestamps, strict=True):
        match (event_id, timestamp):
            case (str(eid), int(ts)) if eid.startswith("evt_") and len(eid) > 4 and ts > 0:
                yield {"event_id": eid, "timestamp": ts}

            case _:
                raise ValueError(f"Invalid event row: {(event_id, timestamp)!r}")

In [18]:
assert list(align_events(["evt_a", "evt_b"], [100, 200])) == [
    {"event_id": "evt_a", "timestamp": 100},
    {"event_id": "evt_b", "timestamp": 200},
]

for ids, times in (
    (["bad"], [100]),
    (["evt_"], [100]),
    (["evt_a"], [0]),
):
    try:
        list(align_events(ids, times))
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for invalid row")

try:
    list(align_events(["evt_a", "evt_b"], [100]))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for unequal streams")

print("Problem 8 tests passed.")

Problem 8 tests passed.


### Solution Notes

This solution combines both topics:

- pattern matching validates each aligned row;
- strict zip validates the relationship between the streams.

That combination is useful in data pipelines where silent truncation can corrupt downstream results.

## Summary of Best Practices

- Put specific `case` blocks before broad fallback cases.
- Use guards for semantic validation that cannot be expressed by the pattern alone.
- Use `_` only as a final wildcard when you truly want a catch-all.
- Prefer custom domain errors such as `ValueError` over accidental errors such as `KeyError`.
- Use `zip(..., strict=True)` when equal lengths are an invariant.
- Avoid pre-checking iterator lengths by converting iterators to lists unless materialization is required.
- Keep functions small: parse one command in one function, parse a program in another.
- Add tests for both success paths and failure paths.